# 🐍 Python od podstaw — Moduł 7: Moduły, pakiety, pip i venv

### Jak Python organizuje kod — i jak korzystać z cudzego

Do tej pory każdy notebook był samodzielną całością. W praktyce programy dzieli się na
wiele plików, a poza tym prawie nigdy nie pisze się wszystkiego od zera — korzysta się z
gotowych bibliotek. Ten moduł to jak to wszystko się ze sobą łączy: `import`, własne
moduły, `pip` i wirtualne środowiska.

## Spis treści

1. [Po co dzielić kod na moduły](#sec1)
2. [`import` — biblioteka standardowa](#sec2)
3. [Tworzenie własnego modułu](#sec3)
4. [`if __name__ == "__main__":`](#sec4)
5. [Pakiety — moduł moduł](#sec5)
6. [`pip` i biblioteki zewnętrzne](#sec6)
7. [Wirtualne środowiska (venv)](#sec7)
8. [Szybki przegląd przydatnych modułów](#sec8)
9. [Ciekawostka: PyPI i „batteries included”](#sec9)
10. [Podsumowanie modułu](#sec10)
11. [Ćwiczenia](#sec11)

---

<a id="sec1"></a>
## 1. Po co dzielić kod na moduły

W module 4 zasada DRY mówiła: nie powtarzaj kodu **w obrębie jednego pliku**. Ta sama
zasada działa w większej skali — nie powtarzaj kodu **między projektami**. Funkcja do
liczenia BMI, którą napisałeś/aś w module 1, przyda się też w innym projekcie. Zamiast
kopiować ją za każdym razem, można ją zapisać raz w osobnym pliku (**module**) i
importować, kiedy jest potrzebna.

<a id="sec2"></a>
## 2. `import` — biblioteka standardowa

Python ma ogromną **bibliotekę standardową** — moduły dostępne od razu, bez instalacji
czegokolwiek. `import` udostępnia zawartość modułu w Twoim kodzie, na kilka sposobów.

In [ ]:
import math
print(math.sqrt(16))     # dostęp przez nazwę modułu
print(math.pi)

import math as m          # alias - krótsza nazwa
print(m.ceil(4.1))        # zaokrąglenie w górę
print(m.floor(4.9))       # zaokrąglenie w dół

from math import sqrt, pi   # import konkretnych nazw - bez prefiksu math.
print(sqrt(25))
print(pi)

# from math import *   # importuje WSZYSTKO - patrz ostrzeżenie niżej

> ⚠️ **Unikaj `from modul import *`**
>
> Import wszystkiego naraz zaśmieca przestrzeń nazw i utrudnia śledzenie, skąd wzięła się dana nazwa - jeśli dwa moduły mają funkcję o tej samej nazwie, jedna po cichu nadpisze drugą. Lepiej importować konkretne nazwy albo cały moduł pod jego (ewentualnie skróconą) nazwą.

<a id="sec3"></a>
## 3. Tworzenie własnego modułu

Moduł to po prostu plik `.py` z definicjami funkcji/klas/zmiennych. W Jupyterze możesz
taki plik stworzyć bezpośrednio z komórki, magicznym poleceniem `%%writefile` (musi być
pierwszą linią komórki) — zapisuje całą resztę komórki do wskazanego pliku.

In [ ]:
%%writefile moj_modul.py
"""Prosty moduł z funkcjami pomocniczymi."""

def przywitaj(imie):
    return f"Cześć, {imie}! To pozdrowienie z modułu."

def podwoj(x):
    return x * 2

In [ ]:
import moj_modul

print(moj_modul.przywitaj("Kamil"))
print(moj_modul.podwoj(21))

> 💡 **Ciekawostka**
>
> `%%writefile` to tzw. *magic command* IPythona/Jupytera (stąd `%%`) - nie jest częścią samego Pythona, działa tylko w notebooku. Poza notebookiem po prostu tworzysz plik `.py` w edytorze kodu i to on jest modułem - `%%writefile` tylko symuluje ten krok, żeby dało się to pokazać w jednym pliku notebooka.

<a id="sec4"></a>
## 4. `if __name__ == "__main__":`

Każdy moduł ma wbudowaną zmienną `__name__`. Gdy plik jest **uruchamiany bezpośrednio**
(np. `python moj_modul.py`), `__name__` ma wartość `"__main__"`. Gdy plik jest
**importowany** z innego pliku, `__name__` ma wartość równą nazwie modułu. Ten idiom
pozwala umieścić w pliku kod, który wykona się tylko przy bezpośrednim uruchomieniu, a
nie przy imporcie.

In [ ]:
%%writefile konwersje.py
"""Moduł z funkcjami konwersji temperatury."""

def celsjusz_na_fahrenheit(c):
    return c * 9 / 5 + 32

def fahrenheit_na_celsjusz(f):
    return (f - 32) * 5 / 9

if __name__ == "__main__":
    # Ten blok wykona się TYLKO przy "python konwersje.py",
    # NIE wykona się przy "import konwersje"
    print("Uruchomiono konwersje.py bezpośrednio - działa tryb demo")
    print(celsjusz_na_fahrenheit(100))

In [ ]:
import konwersje
print(konwersje.celsjusz_na_fahrenheit(20))
# Zauważ: import NIE wypisał "Uruchomiono konwersje.py bezpośrednio..."

# A teraz uruchommy plik bezpośrednio jako skrypt (! uruchamia polecenie w terminalu):
!python konwersje.py

> 💡 **Kiedy to się przydaje**
>
> Dzięki temu idiomowi ten sam plik może być **i** modułem do importowania (w innym programie), **i** samodzielnym skryptem z demem/testami swoich funkcji, uruchamianym wprost z terminala - bardzo częsty wzorzec w prawdziwych projektach Pythona.

<a id="sec5"></a>
## 5. Pakiety — moduł modułów

**Pakiet** to folder zawierający wiele modułów, plus plik `__init__.py` (może być
pusty), który mówi Pythonowi „to jest pakiet, nie zwykły folder”. Pakiety pozwalają
grupować powiązane moduły, np. `narzedzia/tekst.py` i `narzedzia/liczby.py`.

In [ ]:
import os
os.makedirs("moj_pakiet", exist_ok=True)

In [ ]:
%%writefile moj_pakiet/__init__.py
# Ten plik (może być pusty) mówi Pythonowi, że "moj_pakiet" to pakiet

In [ ]:
%%writefile moj_pakiet/tekst.py
def odwroc(tekst):
    return tekst[::-1]

In [ ]:
from moj_pakiet import tekst
print(tekst.odwroc("Python"))

# Albo bezpośredni import konkretnej funkcji:
from moj_pakiet.tekst import odwroc
print(odwroc("Kamil"))

<a id="sec6"></a>
## 6. `pip` i biblioteki zewnętrzne

Biblioteka standardowa (jak `math` czy `random`) jest wbudowana. Ale świat Pythona to
też setki tysięcy **pakietów zewnętrznych** publikowanych na PyPI (*Python Package
Index*) — instaluje się je poleceniem `pip install nazwa_pakietu`, wpisywanym w
terminalu (nie w komórce Pythona!).

In [ ]:
# Znak "!" na początku linii w Jupyterze uruchamia polecenie w terminalu, nie w Pythonie:
!pip --version
!pip list

> 💡 **`requirements.txt`**
>
> Żeby ktoś inny (albo Ty za pół roku) mógł odtworzyć dokładnie te same zależności projektu, zapisuje się je w pliku `requirements.txt` (jedna biblioteka na linię, np. `pandas==2.2.0`). Generuje się go poleceniem `pip freeze > requirements.txt`, a instaluje wszystko naraz przez `pip install -r requirements.txt`.

<a id="sec7"></a>
## 7. Wirtualne środowiska (venv)

Problem: projekt A potrzebuje `pandas==1.5`, a projekt B na tym samym komputerze
potrzebuje `pandas==2.2`. Gdyby biblioteki instalowały się globalnie, jeden z projektów
by się zepsuł. **Wirtualne środowisko** to izolowana, niezależna „kopia” Pythona z
własnym zestawem zainstalowanych pakietów — każdy projekt ma swoje.

Te polecenia uruchamia się **w terminalu**, nie w komórce notebooka:

```bash
python -m venv env          # tworzy nowe środowisko w folderze "env"

# aktywacja (Windows):
env\Scripts\activate

# aktywacja (macOS / Linux):
source env/bin/activate

pip install -r requirements.txt   # instalacja zależności TYLKO w tym środowisku

deactivate                   # wyjście z wirtualnego środowiska
```

> ⚠️ **Anaconda ma swój odpowiednik**
>
> Skoro korzystasz z Jupytera z Anacondy - Anaconda ma własny system środowisk, `conda create -n moje_srodowisko python=3.11` i `conda activate moje_srodowisko`, który działa na tej samej zasadzie co `venv`, tylko innym poleceniem. Nie musisz używać obu naraz - jedno wystarczy.

> 💡 **Ciekawostka**
>
> Problem opisany wyżej ma swoją nazwę: «dependency hell» (piekło zależności) - sytuacja, w której różne projekty (albo różne biblioteki w tym samym projekcie) wymagają niekompatybilnych ze sobą wersji tej samej zależności. Wirtualne środowiska nie rozwiązują tego w 100%, ale znacznie ograniczają skalę problemu.

<a id="sec8"></a>
## 8. Szybki przegląd przydatnych modułów

Kilka modułów standardowych, które prędzej czy później się przydadzą:

| Moduł | Do czego służy |
|---|---|
| `math` | funkcje matematyczne (`sqrt`, `pi`, `ceil`, `floor`...) |
| `random` | liczby losowe, losowy wybór, tasowanie |
| `datetime` | data i czas, obliczenia na datach |
| `os` | operacje na systemie plików, ścieżkach, folderach |
| `sys` | interakcja z interpreterem Pythona (argumenty, ścieżki) |
| `json` | zapis/odczyt danych w formacie JSON |

In [ ]:
from datetime import datetime, timedelta

teraz = datetime.now()
print(teraz)

za_tydzien = teraz + timedelta(days=7)
print(za_tydzien)

import json

dane = {"imie": "Kamil", "jezyki": ["Python", "SQL"]}
with open("dane.json", "w", encoding="utf-8") as plik:
    json.dump(dane, plik, ensure_ascii=False, indent=2)

with open("dane.json", "r", encoding="utf-8") as plik:
    wczytane = json.load(plik)

print(wczytane, type(wczytane))

<a id="sec9"></a>
## 9. Ciekawostka: PyPI i „batteries included”

Filozofia Pythona od początku to „baterie w zestawie” — biblioteka standardowa jest
świadomie bogata, żeby podstawowe zadania (praca z plikami, datami, tekstem, JSON) nie
wymagały żadnej instalacji. A mimo to PyPI liczy **ponad 500 000 pakietów** — od
`requests` (zapytania HTTP) i `pandas` (analiza danych), po biblioteki do dosłownie
wszystkiego, od gier po loty kosmiczne (dosłownie — NASA publikuje pakiety na PyPI).

<a id="sec10"></a>
## 10. Podsumowanie modułu

Po tym module powinno być jasne:

- czym różni się `import x`, `import x as y` i `from x import y`,
- jak stworzyć własny moduł (`.py` z funkcjami) i go zaimportować,
- do czego służy `if __name__ == "__main__":`,
- czym jest pakiet i po co jest `__init__.py`,
- jak zainstalować bibliotekę zewnętrzną (`pip install`) i zapisać zależności
  (`requirements.txt`),
- po co są wirtualne środowiska i jak je stworzyć (`venv` albo `conda`).

To domyka też praktyczną stronę organizacji kodu w większych projektach — od tego
miejsca możesz swobodnie sięgać po dowolną bibliotekę z PyPI, wiedząc, jak ją
bezpiecznie zainstalować i odizolować od innych projektów.

<a id="sec11"></a>
## 11. Ćwiczenia

Część zadań to praktyczne użycie modułów standardowych, część - tworzenie własnych
plików `.py` przez `%%writefile`, dokładnie jak w sekcjach teoretycznych.

> 📝 **Ćwiczenie 1: `math` w praktyce**
>
> Zaimportuj moduł `math`. Dla liczby `x = 47.3` wypisz: pierwiastek kwadratowy, zaokrąglenie w górę i w dół, oraz wartość `math.pi` zaokrągloną do 3 miejsc po przecinku.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import math

x = 47.3
print(math.sqrt(x))
print(math.ceil(x))
print(math.floor(x))
print(round(math.pi, 3))
```
</details>

> 📝 **Ćwiczenie 2: `random` w praktyce**
>
> Zaimportuj `random`. Wylosuj liczbę całkowitą od 1 do 100, wylosuj jeden element z listy `["orzeł", "reszka"]`, oraz przetasuj (`.shuffle()`) listę `[1, 2, 3, 4, 5]` i wypisz ją po przetasowaniu.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import random

print(random.randint(1, 100))
print(random.choice(["orzeł", "reszka"]))

liczby = [1, 2, 3, 4, 5]
random.shuffle(liczby)
print(liczby)
```
</details>

> 📝 **Ćwiczenie 3: Ile dni do wakacji**
>
> Za pomocą `datetime` policz, ile dni zostało od dziś (`datetime.now()`) do konkretnej daty w przyszłości, np. `datetime(2026, 12, 24)` (Wigilia). Podpowiedź: odejmowanie dwóch obiektów `datetime` daje `timedelta`, który ma atrybut `.days`.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
from datetime import datetime

teraz = datetime.now()
wigilia = datetime(2026, 12, 24)

roznica = wigilia - teraz
print(f"Dni do Wigilii: {roznica.days}")
```
</details>

> 📝 **Ćwiczenie 4: Własny moduł z konwerterem walut**
>
> Użyj `%%writefile`, żeby stworzyć plik `waluty.py` z funkcją `pln_na_eur(kwota, kurs=4.3)`, zwracającą kwotę podzieloną przez kurs. W kolejnej komórce zaimportuj ten moduł i przelicz 500 zł na euro.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# Komórka 1:
# %%writefile waluty.py
# def pln_na_eur(kwota, kurs=4.3):
#     return kwota / kurs

# Komórka 2:
# import waluty
# print(waluty.pln_na_eur(500))
```

To ćwiczenie wymaga dwóch komórek (magic `%%writefile` musi być jedyną treścią swojej komórki) - powyższe rozwiązanie pokazuje treść obu, ale w praktyce rozdziel je na dwie osobne komórki kodu, tak jak w sekcji 3.
</details>

> 📝 **Ćwiczenie 5: `__name__ == "__main__"` w akcji**
>
> Dopisz do pliku `waluty.py` z poprzedniego ćwiczenia (albo stwórz go od nowa) blok `if __name__ == "__main__":`, który wypisuje wynik przeliczenia 1000 zł na euro. Sprawdź, że przy `import waluty` ten blok się NIE wykonuje, a przy `!python waluty.py` - wykonuje się.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# %%writefile waluty.py
# def pln_na_eur(kwota, kurs=4.3):
#     return kwota / kurs
#
# if __name__ == "__main__":
#     print(f"1000 zł to {pln_na_eur(1000):.2f} euro")

# import waluty          # nic dodatkowego się nie wypisze
# !python waluty.py       # to wypisze wynik przeliczenia
```

Podpowiedź: jeśli moduł `waluty` był już raz zaimportowany w tej sesji Jupytera, ponowny `import waluty` nic nie zrobi (Python cache'uje moduły) - żeby zobaczyć zmiany po edycji pliku, użyj `importlib.reload(waluty)` albo zrestartuj kernel.
</details>

> 📝 **Ćwiczenie 6: Zapis i odczyt JSON**
>
> Stwórz słownik z danymi o sobie (imię, wiek, ulubione języki programowania jako lista). Zapisz go do pliku `profil.json` przez `json.dump()`, a potem wczytaj ponownie przez `json.load()` i sprawdź `type()` wczytanej wartości.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
import json

profil = {
    "imie": "Kamil",
    "wiek": 30,
    "jezyki": ["Python", "SQL", "JavaScript"],
}

with open("profil.json", "w", encoding="utf-8") as plik:
    json.dump(profil, plik, ensure_ascii=False, indent=2)

with open("profil.json", "r", encoding="utf-8") as plik:
    wczytany_profil = json.load(plik)

print(wczytany_profil)
print(type(wczytany_profil))
```
</details>

> 📝 **Ćwiczenie 7: Plik requirements.txt**
>
> Za pomocą `%%writefile` (albo zwykłego `open()`/`.write()`) stwórz plik `requirements.txt` dla projektu, który korzysta z: `pandas` w wersji dokładnie `2.2.0`, `requests` w dowolnej wersji, oraz `numpy` w wersji co najmniej `1.24` (operator `>=`).

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
with open("requirements.txt", "w", encoding="utf-8") as plik:
    plik.write("pandas==2.2.0\n")
    plik.write("requests\n")
    plik.write("numpy>=1.24\n")

with open("requirements.txt", "r", encoding="utf-8") as plik:
    print(plik.read())
```

W prawdziwym projekcie ten plik odtwarza się poleceniem `pip install -r requirements.txt`, najlepiej wewnątrz aktywowanego wirtualnego środowiska.
</details>

> 🔥 **Ćwiczenie 8 (wyzwanie): Własny mini-pakiet**
>
> Stwórz pakiet `moje_narzedzia` (folder + `__init__.py`) z dwoma modułami: `tekst.py` z funkcją `czy_palindrom(s)` (sprawdza, czy string czytany od tyłu jest taki sam, po zamianie na małe litery) oraz `liczby.py` z funkcją `czy_pierwsza(n)` (sprawdza, czy liczba jest pierwsza). Zaimportuj oba moduły i przetestuj obie funkcje na kilku przykładach.

<details>
<summary><b>👉 Kliknij, żeby zobaczyć przykładowe rozwiązanie</b></summary>

```python
# Komórka 1:
# import os
# os.makedirs("moje_narzedzia", exist_ok=True)

# Komórka 2:
# %%writefile moje_narzedzia/__init__.py
# # pusty plik oznaczający pakiet

# Komórka 3:
# %%writefile moje_narzedzia/tekst.py
# def czy_palindrom(s):
#     s = s.lower()
#     return s == s[::-1]

# Komórka 4:
# %%writefile moje_narzedzia/liczby.py
# def czy_pierwsza(n):
#     if n < 2:
#         return False
#     for dzielnik in range(2, int(n ** 0.5) + 1):
#         if n % dzielnik == 0:
#             return False
#     return True

# Komórka 5:
# from moje_narzedzia import tekst, liczby
#
# print(tekst.czy_palindrom("kajak"))
# print(tekst.czy_palindrom("Python"))
# print(liczby.czy_pierwsza(17))
# print(liczby.czy_pierwsza(18))
```

Podpowiedź: każdy `%%writefile` musi być jedyną zawartością swojej komórki - w prawdziwym notebooku to 5 osobnych komórek uruchamianych po kolei, dokładnie jak w sekcji 5 z modułem `tekst`.
</details>

---

### Co dalej?

Gratulacje — to koniec siedmiu modułów: od `print()` po organizację kodu w moduły,
pakiety i izolowane środowiska. Masz teraz komplet narzędzi, żeby zacząć realny
projekt albo sięgnąć po dowolną bibliotekę z PyPI (np. `pandas` do analizy danych,
`requests` do pracy z API, `matplotlib` do wykresów) i wiesz, jak bezpiecznie ją
zainstalować.